# YouTube lecture queue → deck packages (Colab v0.1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herndoch/pathology-hub-dev/blob/master/notebooks/YouTube_Lecture_Queue_Colab_v0_1.ipynb)


**One video per Colab runtime.** YouTube bot-checks stick to a warm session/IP. Restarting (Disconnect **and delete** runtime) between videos is the no-cookies path that worked for Cipriani.

**Workflow**
1. Cell **1a** = URL list (`QUEUE`) — add all videos once
2. Cell **1b** = `INDEX` only — which URL this runtime runs
3. **Runtime → Run all**
4. Copy the printed `PACKAGE_ID` into chat
5. **Runtime → Disconnect and delete runtime**
6. Reconnect, set `INDEX += 1`, Run all again

Secrets: `OPEN_AI_KEY_01` (Whisper). GCS: Drive-mounted SA json or `GOOGLE_APPLICATION_CREDENTIALS`.


In [ ]:
# ========== CELL 1a — URL QUEUE (add rows here) ==========
# One row = one lecture. Captions-first; Whisper if needed.
# (youtube_url, browse_root, optional_package_id_or_None)

QUEUE = [
    ("https://www.youtube.com/watch?v=rCdaaTDesPQ", "Breast", None),  # Damron breast board review (may be age-restricted)
    ("https://www.youtube.com/watch?v=1WuhaGCtj4k", "BST", None),      # Gardner 21 classic cases
    # ("https://www.youtube.com/watch?v=NOmOHTh-vtY", "BST", None),   # Cipriani — already done
    # Add more rows here…
]

SKIP_FRAMES = True
PREFER_CAPTIONS = True
WHISPER_MAX_MB = 24.0
WHISPER_CHUNK_SEC = 600
UPLOAD_FRAMES_ASSETS = False
DRY_RUN = False  # True = print GCS destinations only


In [ ]:
# ========== CELL 1b — PICK ONE (edit INDEX only between runtimes) ==========
# After success: Disconnect and delete runtime → bump INDEX → reconnect → Run all

INDEX = 0

assert 0 <= INDEX < len(QUEUE), f"INDEX {INDEX} out of range 0..{len(QUEUE)-1}"
YOUTUBE_URL, ROOT, PACKAGE_ID_OVERRIDE = QUEUE[INDEX]
print(f"INDEX {INDEX}/{len(QUEUE)-1}")
print("URL ", YOUTUBE_URL)
print("ROOT", ROOT)
print("PKG ", PACKAGE_ID_OVERRIDE)


## Pipeline (run all — do not edit below unless debugging)


In [ ]:
# Cell 2 — installs, auth, helpers
!pip -q install -U yt-dlp openai google-cloud-storage youtube-transcript-api

from google.colab import auth, userdata
auth.authenticate_user()

import json, os, re, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import parse_qs, urlparse
from urllib.request import urlopen

from google.cloud import storage
from openai import OpenAI

PROJECT = "pathology-annotation-project"
HUB_BUCKET = "pathology_hub"
VIDEO_BUCKET = "pathology-hub-0"
ASSET_PREFIX = "_asset_library/lectures/"

def load_openai_key():
    for name in ("OPEN_AI_KEY_01", "OPENAI_API_KEY", "OPEN_AI_KEY"):
        try:
            k = userdata.get(name)
            if k:
                print("OpenAI key from Colab secret:", name)
                return k
        except Exception:
            pass
    k = os.environ.get("OPENAI_API_KEY")
    assert k, "Set Colab secret OPEN_AI_KEY_01 (needed only if Whisper runs)"
    return k

os.environ["OPENAI_API_KEY"] = load_openai_key()
oai = OpenAI()
gcs = storage.Client(project=PROJECT)
hub = gcs.bucket(HUB_BUCKET)
assets = gcs.bucket(VIDEO_BUCKET)

WORK = Path("/content/yt_ingest_work")
WORK.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def slugify(text: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9]+", "_", text.strip()).strip("_").lower()
    return s or "lecture"

def extract_youtube_id(url: str) -> str:
    u = urlparse(url)
    if "youtu.be" in (u.netloc or ""):
        return u.path.strip("/").split("/")[0].split("?")[0]
    qs = parse_qs(u.query)
    if "v" in qs and qs["v"]:
        return qs["v"][0]
    m = re.search(r"(?:embed|shorts|live)/([A-Za-z0-9_-]{6,})", url)
    if m:
        return m.group(1)
    raise ValueError(url)

def youtube_watch_url(video_id: str) -> str:
    return f"https://www.youtube.com/watch?v={video_id}"

def make_youtube_time_url(video_id: str, start) -> Optional[str]:
    try:
        s = int(float(start))
    except (TypeError, ValueError):
        return None
    return f"https://www.youtube.com/watch?v={video_id}&t={max(0, s)}s"

def run(cmd):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True, capture_output=True, text=True)

VIDEO_ID = extract_youtube_id(YOUTUBE_URL)
YT_URL = youtube_watch_url(VIDEO_ID)
print("VIDEO_ID", VIDEO_ID)


In [ ]:
# Cell 3 — metadata + captions-first OR audio-only

def fetch_meta(url: str) -> dict:
    try:
        proc = run(["yt-dlp", "--no-playlist", "--dump-single-json", "--skip-download", url])
        return json.loads(proc.stdout)
    except subprocess.CalledProcessError as exc:
        err = (exc.stderr or "")
        print("yt-dlp meta failed; oEmbed fallback
", err[-500:])
        if "confirm your age" in err.lower() or "age" in err.lower():
            print("NOTE: age-restricted — captions may still work; media download may need cookies")
        oe = json.loads(urlopen(f"https://www.youtube.com/oembed?url={url}&format=json", timeout=30).read())
        return {
            "id": VIDEO_ID,
            "title": oe.get("title"),
            "uploader": oe.get("author_name"),
            "channel": oe.get("author_name"),
            "duration": None,
            "_meta_source": "oembed_fallback",
        }

def fetch_caption_segments(video_id: str) -> list:
    from youtube_transcript_api import YouTubeTranscriptApi
    fetched = YouTubeTranscriptApi().fetch(video_id)
    out = []
    for i, s in enumerate(fetched):
        if hasattr(s, "text"):
            text = (s.text or "").strip()
            start = float(s.start or 0.0)
            dur = float(getattr(s, "duration", 0.0) or 0.0)
        else:
            text = (s.get("text") or "").strip()
            start = float(s.get("start") or 0.0)
            dur = float(s.get("duration") or 0.0)
        if not text:
            continue
        out.append({"id": i, "start": start, "end": start + max(dur, 0.01), "text": text})
    return out

meta = fetch_meta(YOUTUBE_URL)
TITLE = meta.get("title") or f"YouTube {VIDEO_ID}"
UPLOADER = meta.get("uploader") or meta.get("channel") or ""
DURATION = meta.get("duration")
print(json.dumps({"title": TITLE, "uploader": UPLOADER, "duration": DURATION}, indent=2))

SEGMENTS = []
TRANSCRIPT_SOURCE = None
AUDIO = None

if PREFER_CAPTIONS:
    try:
        SEGMENTS = fetch_caption_segments(VIDEO_ID)
        TRANSCRIPT_SOURCE = "youtube_captions"
        print(f"captions OK segments={len(SEGMENTS)}")
        print("sample:", SEGMENTS[0] if SEGMENTS else None)
    except Exception as e:
        print("captions failed:", type(e).__name__, e)

if not SEGMENTS:
    audio_tmpl = str(WORK / "audio.%(ext)s")
    try:
        run([
            "yt-dlp", "--no-playlist",
            "-f", "bestaudio[ext=m4a]/bestaudio/best",
            "-o", audio_tmpl,
            "--extract-audio", "--audio-format", "mp3", "--audio-quality", "0",
            YOUTUBE_URL,
        ])
    except subprocess.CalledProcessError as exc:
        err = (exc.stderr or "")[-800:]
        raise RuntimeError(
            "Audio download failed (bot or age gate). "
            "Disconnect+delete runtime and retry, or skip this URL. "
            "Age-restricted titles may need cookies (not used here).\n" + err
        ) from exc
    AUDIO = next(p for p in WORK.glob("audio.*") if p.suffix.lower() in {".mp3", ".m4a", ".webm", ".opus", ".wav"})
    print("AUDIO", AUDIO, f"{AUDIO.stat().st_size/1e6:.1f} MB")

if DURATION is None and AUDIO is not None:
    try:
        proc = run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                    "-of", "default=noprint_wrappers=1:nokey=1", str(AUDIO)])
        DURATION = float(proc.stdout.strip())
    except Exception as e:
        print("WARN duration probe", e)


In [ ]:
# Cell 4 — Whisper if needed (recompress + chunk for >24MB)

def whisper_file(path: Path) -> list:
    with path.open("rb") as f:
        resp = oai.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )
    data = resp.model_dump() if hasattr(resp, "model_dump") else dict(resp)
    out = []
    for i, s in enumerate(data.get("segments") or []):
        text = (s.get("text") or "").strip()
        if not text:
            continue
        out.append({
            "id": i,
            "start": float(s.get("start") or 0.0),
            "end": float(s.get("end") or s.get("start") or 0.0),
            "text": text,
        })
    return out

def prepare_audio_under_limit(audio: Path, max_mb: float = WHISPER_MAX_MB) -> Path:
    size_mb = audio.stat().st_size / 1e6
    if size_mb <= max_mb:
        return audio
    print(f"audio {size_mb:.1f} MB > {max_mb} MB — recompressing")
    out = audio
    for br in ("64k", "48k", "32k", "24k"):
        out = WORK / f"audio_{br}.mp3"
        run(["ffmpeg", "-y", "-i", str(audio), "-ac", "1", "-ar", "16000", "-b:a", br, str(out)])
        sm = out.stat().st_size / 1e6
        print(f"  {br} -> {sm:.1f} MB")
        if sm <= max_mb:
            return out
    return out

def whisper_chunked(audio: Path, chunk_sec: int = WHISPER_CHUNK_SEC, max_mb: float = WHISPER_MAX_MB) -> list:
    chunk_dir = WORK / "whisper_chunks"
    if chunk_dir.exists():
        shutil.rmtree(chunk_dir)
    chunk_dir.mkdir(parents=True)
    pattern = str(chunk_dir / "chunk_%03d.mp3")
    run([
        "ffmpeg", "-y", "-i", str(audio),
        "-f", "segment", "-segment_time", str(int(chunk_sec)),
        "-reset_timestamps", "1",
        "-ac", "1", "-ar", "16000", "-b:a", "32k",
        pattern,
    ])
    parts = sorted(chunk_dir.glob("chunk_*.mp3"))
    print(f"whisper chunks: {len(parts)} x ~{chunk_sec}s")
    all_segs = []
    for i, part in enumerate(parts):
        print(f"  whisper chunk {i}: {part.name} ({part.stat().st_size/1e6:.1f} MB)")
        segs = whisper_file(part)
        offset = i * float(chunk_sec)
        for s in segs:
            s["start"] += offset
            s["end"] += offset
            s["id"] = len(all_segs)
            all_segs.append(s)
    return all_segs

if SEGMENTS:
    print(f"skip Whisper; using {TRANSCRIPT_SOURCE} ({len(SEGMENTS)} segments)")
else:
    assert AUDIO is not None
    audio = prepare_audio_under_limit(AUDIO, WHISPER_MAX_MB)
    size_mb = audio.stat().st_size / 1e6
    if size_mb <= WHISPER_MAX_MB:
        print(f"Whisper single-file {audio.name} ({size_mb:.1f} MB)")
        SEGMENTS = whisper_file(audio)
    else:
        print(f"still {size_mb:.1f} MB — chunked Whisper")
        SEGMENTS = whisper_chunked(audio, WHISPER_CHUNK_SEC, WHISPER_MAX_MB)
    TRANSCRIPT_SOURCE = "whisper_audio"
    print("segments", len(SEGMENTS))
    print("sample:", SEGMENTS[0] if SEGMENTS else None)

assert SEGMENTS, "Empty transcript"
if DURATION is None:
    DURATION = float(SEGMENTS[-1]["end"])
print("TRANSCRIPT_SOURCE", TRANSCRIPT_SOURCE, "DURATION", DURATION)


In [ ]:
# Cell 5 — build package + upload

display_title = f"{TITLE} ({UPLOADER})" if UPLOADER else TITLE
PACKAGE_ID = PACKAGE_ID_OVERRIDE or (
    f"yt_{slugify(UPLOADER)[:24]}_{slugify(TITLE)[:48]}_{VIDEO_ID.lower()}_v0_1"
)
if len(PACKAGE_ID) > 90:
    PACKAGE_ID = f"yt_{VIDEO_ID.lower()}_v0_1"

PKG = WORK / "packages" / PACKAGE_ID
if PKG.exists():
    shutil.rmtree(PKG)
PKG.mkdir(parents=True)
(PKG / "frames").mkdir(exist_ok=True)

video_url = YT_URL
seg_rows = []
for s in SEGMENTS:
    start, end = float(s["start"]), float(s["end"])
    seg_rows.append({
        "schema_version": "lecture_deck_segment.v0_1",
        "package_id": PACKAGE_ID,
        "segment_id": f"{PACKAGE_ID}::seg_{int(s['id']):05d}",
        "start_sec": start,
        "end_sec": end,
        "text": s["text"],
        "language": "en",
        "video_id": VIDEO_ID,
        "video_url": video_url,
        "video_time_url": make_youtube_time_url(VIDEO_ID, start),
        "raw_source_gcs_uri": None,
        "raw_source_join_basis": "youtube_watch_url",
        "youtube_url": YT_URL,
        "primary_tag": None,
        "tag_status": "untagged",
        "root": ROOT,
        "indexable": False,
        "source_format": f"youtube_{TRANSCRIPT_SOURCE}_colab_queue_v0",
    })

manifest = {
    "schema_version": "lecture_deck_package.v0_1",
    "package_id": PACKAGE_ID,
    "title": display_title,
    "root": ROOT,
    "source_format": "youtube_ingest_colab_queue_v0_1",
    "youtube_url": YT_URL,
    "youtube_video_id": VIDEO_ID,
    "video_file_declared": None,
    "duration_seconds": float(DURATION) if DURATION is not None else None,
    "video_id": VIDEO_ID,
    "raw_source_gcs_uri": None,
    "video_url": video_url,
    "raw_source_join_basis": "youtube_watch_url",
    "playback": "youtube",
    "transcript_source": TRANSCRIPT_SOURCE,
    "queue_index": INDEX,
    "counts": {
        "segments": len(seg_rows),
        "frames": 0,
        "segments_with_video_time_url": sum(1 for s in seg_rows if s.get("video_time_url")),
        "frames_with_video_time_url": 0,
        "canonical_mp4_present": False,
    },
    "created_at_utc": utc_now(),
    "known_limitations": [
        "Playback is YouTube &t= (not GCS MP4 #t=).",
        "Frames skipped (audio/captions-only queue notebook).",
        "chunks_indexable.jsonl empty until agent semantic gate.",
    ],
    "source_url": YOUTUBE_URL,
}

audit = {
    "schema_version": "lecture_deck_youtube_colab_queue_audit.v0_1",
    "created_at_utc": utc_now(),
    "package_id": PACKAGE_ID,
    "queue_index": INDEX,
    "input_paths": [YOUTUBE_URL],
    "output_paths": [str(PKG)],
    "counts": manifest["counts"],
    "transcript_source": TRANSCRIPT_SOURCE,
    "known_limitations": manifest["known_limitations"],
}

(PKG / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
with (PKG / "segments.jsonl").open("w") as f:
    for row in seg_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
(PKG / "frames.jsonl").write_text("")
(PKG / "audit.json").write_text(json.dumps(audit, indent=2) + "\n")
(PKG / "segments_indexable.jsonl").write_text("")
(PKG / "chunks_indexable.jsonl").write_text("")

print("PACKAGE_ID", PACKAGE_ID)
print("PKG", PKG)
print("segments", len(seg_rows))

uploaded = []
prefix = f"02_normalized/lectures/deck_packages/{PACKAGE_ID}/"
for name in (
    "manifest.json", "segments.jsonl", "frames.jsonl", "audit.json",
    "segments_indexable.jsonl", "chunks_indexable.jsonl",
):
    path = PKG / name
    if not path.is_file():
        continue
    dest = prefix + name
    if DRY_RUN:
        print("DRY", f"gs://{HUB_BUCKET}/{dest}")
    else:
        hub.blob(dest).upload_from_filename(str(path))
        print("uploaded", f"gs://{HUB_BUCKET}/{dest}")
    uploaded.append(f"gs://{HUB_BUCKET}/{dest}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
gcs_audit = {
    "schema_version": "lecture_deck_youtube_colab_queue_upload_audit.v0_1",
    "created_at_utc": utc_now(),
    "package_id": PACKAGE_ID,
    "queue_index": INDEX,
    "input_paths": [YOUTUBE_URL],
    "output_paths": [f"gs://{HUB_BUCKET}/{prefix}"],
    "counts": {**manifest["counts"], "sidecar_files_uploaded": len(uploaded), "dry_run": DRY_RUN},
    "uploaded": uploaded,
    "transcript_source": TRANSCRIPT_SOURCE,
    "known_limitations": manifest["known_limitations"],
}
audit_key = f"06_audits/lectures/deck_packages/youtube_colab_queue_{stamp}/audit.json"
if not DRY_RUN:
    hub.blob(audit_key).upload_from_string(json.dumps(gcs_audit, indent=2) + "\n", content_type="application/json")
print("audit =>", f"gs://{HUB_BUCKET}/{audit_key}")
print()
print("=" * 60)
print("PACKAGE_ID for agent:", PACKAGE_ID)
print(f"ROOT={ROOT}  INDEX={INDEX}  next INDEX={INDEX+1 if INDEX+1 < len(QUEUE) else 'DONE'}")
print("Tell the agent: gate", ROOT, "package", PACKAGE_ID)
print("Then: Runtime → Disconnect and delete runtime → bump INDEX → reconnect → Run all")
print("=" * 60)


### After success
1. Paste `PACKAGE_ID` into the agent chat  
2. **Disconnect and delete runtime**  
3. Set `INDEX = INDEX + 1` in Cell 1  
4. Connect → Run all  

Agent will gate with the package `ROOT` (Breast / BST / …) and rebuild the lecture vector when you say so.
